# CropFormer Mask Prediction on Replica
This notebook runs CropFormer (Mask2Former + HorNet) on the Replica dataset RGB frames.

**Runtime**: Make sure you select a GPU runtime: Runtime → Change runtime type → T4 GPU

## 1. Mount Google Drive & extract data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!mkdir -p /content/data
!tar xzf /content/drive/MyDrive/ML3D/replica_color.tar.gz -C /content/data/
!ls /content/data/replica/

## 2. Install dependencies

In [ ]:
!pip install torch torchvision torchaudio
!pip install opencv-python tqdm numpy timm

In [ ]:
# Install detectron2
!git clone https://github.com/facebookresearch/detectron2.git /content/detectron2
!pip install -e /content/detectron2

In [ ]:
# Clone Entity repo and copy CropFormer into detectron2
!git clone https://github.com/qqlu/Entity.git /content/Entity
!cp -r /content/Entity/Entityv2/CropFormer /content/detectron2/projects/

In [ ]:
# Build the deformable attention CUDA op
%cd /content/detectron2/projects/CropFormer/mask2former/modeling/pixel_decoder/ops
!sh make.sh
%cd /content

## 3. Download CropFormer checkpoint
The model weights are hosted on HuggingFace.

In [ ]:
!pip install huggingface_hub
from huggingface_hub import hf_hub_download

# Download the Mask2Former HorNet 3x checkpoint
ckpt_path = hf_hub_download(
    repo_id="qqlu1992/Adobe_EntitySeg",
    filename="CropFormer_model/Entity_Segmentation/Mask2Former_hornet_3x/model_final_entityv2_hornet_3x.pth",
    repo_type="dataset"
)
print(f"Checkpoint downloaded to: {ckpt_path}")

## 4. Run CropFormer on all Replica scenes

In [ ]:
import glob
import os

scenes = sorted([d for d in os.listdir('/content/data/replica') if os.path.isdir(f'/content/data/replica/{d}')])
print(f"Found {len(scenes)} scenes: {scenes}")

In [ ]:
CONFIG_FILE = "/content/detectron2/projects/CropFormer/configs/entityv2/entity_segmentation/mask2former_hornet_3x.yaml"
CROPFORMER_DIR = "/content/detectron2/projects/CropFormer/demo_cropformer"

seq_name_list = '+'.join(scenes)

!cd {CROPFORMER_DIR} && python mask_predict.py \
    --config-file {CONFIG_FILE} \
    --root /content/data/replica \
    --image_path_pattern "color/*.jpg" \
    --dataset replica \
    --seq_name_list {seq_name_list} \
    --opts MODEL.WEIGHTS {ckpt_path}

## 5. Verify output and check a sample mask

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Check output structure
for scene in scenes:
    mask_dir = f'/content/data/replica/{scene}/output/mask'
    if os.path.exists(mask_dir):
        n_masks = len(os.listdir(mask_dir))
        print(f"{scene}: {n_masks} mask files")
    else:
        print(f"{scene}: NO OUTPUT")

# Visualize one mask
sample_mask = cv2.imread(f'/content/data/replica/office0/output/mask/0.png', cv2.IMREAD_UNCHANGED)
sample_rgb = cv2.imread(f'/content/data/replica/office0/color/0.jpg')
sample_rgb = cv2.cvtColor(sample_rgb, cv2.COLOR_BGR2RGB)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.imshow(sample_rgb)
ax1.set_title('RGB')
ax2.imshow(sample_mask, cmap='tab20')
ax2.set_title(f'CropFormer masks ({sample_mask.max()} instances)')
plt.show()

## 6. Package masks and save to Drive

In [ ]:
!cd /content/data && tar czf /content/replica_masks.tar.gz replica/*/output/
!cp /content/replica_masks.tar.gz /content/drive/MyDrive/ML3D/
!ls -lh /content/drive/MyDrive/ML3D/replica_masks.tar.gz
print("Done! Download replica_masks.tar.gz from your Drive.")